# 00 · Colab kurulumu ve deney koşuları

Bu notebook depoyu Colab'a kurar, testleri çalıştırır ve deneyleri başlatır.

1. **Çalışma zamanı:** Menüden *Runtime → Change runtime type → T4 GPU* seç. Aylık deney CPU'da da hızlıdır; günlük deneyler GPU ile çok daha kısa sürer.
2. Hücreleri yukarıdan aşağıya sırayla çalıştır.
3. Colab oturumu kapanınca `/content` altındaki her şey silinir. Sonuçları saklamak için en alttaki **Drive'a kopyala** hücresini çalıştır.

## 1. Depoyu getir

In [ ]:
import os

REPO_URL = "https://github.com/salihcengiz/sector-spillover.git"
ROOT = "/content/sector-spillover"

if os.path.isdir(ROOT):
    !git -C {ROOT} pull --ff-only     # depo zaten varsa son sürümü çek
else:
    !git clone {REPO_URL} {ROOT}
%cd {ROOT}
!git log --oneline -3

## 2. Kurulum

torch, numpy, pandas, scikit-learn, scipy, matplotlib, pyyaml ve pytest Colab'da hazır geliyor. Yalnızca projenin kendi paketi (`spillover`) kuruluyor; `requirements.txt` bilerek kurulmuyor.

In [ ]:
!pip install -q -e .

import sys, numpy, pandas, torch, yaml
print("python ", sys.version.split()[0])
print("numpy  ", numpy.__version__, "| pandas", pandas.__version__, "| torch", torch.__version__)
print("cihaz  ", "cuda: " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu (GPU seçilmemiş)")

## 3. Testler

Testlerin hepsi geçmeli. Model ve eğitim testleri (`test_models.py`, `test_experiment.py` içindeki LSTM testi) ilk kez burada torch ile çalışıyor.

In [ ]:
!python -m pytest -q

## 4. Hızlı deneme (`--quick`: tek seed, en fazla 2 epoch)

Uçtan uca akışın hatasız çalıştığını birkaç dakikada doğrular. Bu koşunun sonuçları anlamlı değildir.

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_monthly_b.yaml --quick

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_daily_crisis.yaml --quick --folds covid

## 5. Tam koşular

Her model 5 seed ile eğitilir ve seed'lerin ortalaması da ayrı bir model olarak raporlanır.

- `lstm_monthly_b`: aylık, B düzeni (train ≤ 2010, val 2011–2016, test 2017 → 2026-09). CPU'da ~5 dk.
- `lstm_monthly_a`: aylık, A düzeni (walk-forward). Aynı test dönemi; her yıl başında modeller o güne kadarki veriyle yeniden eğitilir (10 katman). CPU'da ~45–60 dk, GPU'da daha kısa.
- `lstm_daily_crisis`: günlük, hocanın düzeni. COVID, SVB ve son 1 yıl için üç ayrı katman; her katmanda modeller yeniden eğitilir. GPU gerekir.
- `lstm_daily_b`: günlük, B düzeni. Ön analizle karşılaştırmak için, isteğe bağlı.

Birden çok katmanlı koşularda bütün katmanların test tahminleri ayrıca tek seri olarak değerlendirilir (`pooled`).

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_monthly_b.yaml

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_monthly_a.yaml

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_daily_crisis.yaml

In [ ]:
# İsteğe bağlı:
# !python scripts/run_experiment.py --config configs/lstm_daily_b.yaml

## 6. Sonuçlara hızlı bakış

Önce fonksiyonları tanımlayan hücreyi çalıştır, sonra istediğin deneyin hücresini. Her koşunun klasörü `outputs/runs/<zaman>_<ad>/`. `show`, adı verilen deneyin en son tam koşusunu düz metin olarak yazdırır; çıktıyı olduğu gibi kopyalayıp paylaşabilirsin.

- `oos_r2`: train ortalamasına göre test R² (49 sektörün ortalaması). `pooled` sütunu bütün katmanların birlikte değerlendirilmesidir; tek tek yıllar/pencereler az örnek içerdiği için ana sonuç odur.
- `rel_mse`: HAR-kendi'ye göre göreli MSE; 1'in altı daha iyi.
- DM: `istatistik (p)`. İstatistik negatifse ilk model daha iyi; p < 0.05 ise fark anlamlı.

In [ ]:
import glob, os
import pandas as pd

def latest_run(name):
    runs = sorted(glob.glob(f"outputs/runs/*_{name}"))
    return runs[-1] if runs else None

def show(name):
    run = latest_run(name)
    if run is None:
        print(f"{name}: bu oturumda tam koşu yok (önce 5. bölümdeki hücresini çalıştır).")
        return
    print(run, "\n")
    summary = pd.read_csv(f"{run}/summary_all.csv")
    folds = list(dict.fromkeys(summary["fold"]))
    for metric, title in (("oos_r2", "OOS R²"), ("rel_mse", "HAR-kendi'ye göre göreli MSE")):
        table = summary.pivot(index="model", columns="fold", values=metric)[folds]
        print(f"{title}:\n{table.round(3).to_string()}\n")
    rows = {}
    for fold in folds:
        dm = pd.read_csv(f"{run}/{fold}/dm.csv")
        dm = dm[(dm.sector == "ALL") & (dm.loss == "mse")]
        for _, r in dm.iterrows():
            rows.setdefault(f"{r.model} vs {r.reference}", {})[fold] = f"{r.stat:6.2f} ({r.p_value:.3f})"
    print(f"DM testi, MSE, 49 sektör ortalaması — istatistik (p):\n{pd.DataFrame(rows).T[folds].to_string()}\n")
    if os.path.exists(f"{run}/slices.csv"):
        sl = pd.read_csv(f"{run}/slices.csv")
        n = sl.groupby("slice")["n"].first()
        table = sl.pivot(index="model", columns="slice", values="rel_mse")[list(n.index)]
        table.columns = [f"{c} (n={n[c]})" for c in table.columns]
        print(f"Dönem kesitleri, HAR-kendi'ye göre göreli MSE:\n{table.round(3).to_string()}")

In [ ]:
show("lstm_monthly_b")

In [ ]:
show("lstm_monthly_a")

In [ ]:
show("lstm_daily_crisis")

## 7. Sonuçları Google Drive'a kopyala

Drive'da `MyDrive/sector-spillover/outputs/` klasörüne kopyalar. Daha önce kopyalanmış koşuların üzerine yazmaz.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/sector-spillover/outputs
!rsync -a --ignore-existing outputs/ /content/drive/MyDrive/sector-spillover/outputs/
!ls /content/drive/MyDrive/sector-spillover/outputs/runs